# 02 — TMDB Enrichment

Enrich the MovieLens 100K movie list with metadata from the TMDB API: actors, directors, and genres.

**Pipeline:**
1. Load `u.item` from MovieLens
2. Parse title and year from the MovieLens title format
3. Search TMDB automatically for each movie
4. Identify and export failed matches
5. Load manual mapping for failed matches
6. Re-run enrichment using manual TMDB IDs
7. Save final enriched dataset

> See `docs/02_enrichment_approach.md` for the full design rationale.

## 0. Setup

In [24]:
import os
import json
import time
import requests
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()

TMDB_API_KEY = os.environ["TMDB_API_KEY"]
TMDB_BASE = "https://api.themoviedb.org/3"

def tmdb_get(endpoint, params=None):
    url = f"{TMDB_BASE}/{endpoint}"
    response = requests.get(url, params={"api_key": TMDB_API_KEY, **(params or {})})
    response.raise_for_status()
    return response.json()

print("Ready.")

Ready.


## 1. Explore — Genre List
What genres does TMDB know about? This is a single call — we'll use this to map genre IDs to names later.

In [2]:
genres = tmdb_get("genre/movie/list")
print(json.dumps(genres, indent=2))

{
  "genres": [
    {
      "id": 28,
      "name": "Action"
    },
    {
      "id": 12,
      "name": "Adventure"
    },
    {
      "id": 16,
      "name": "Animation"
    },
    {
      "id": 35,
      "name": "Comedy"
    },
    {
      "id": 80,
      "name": "Crime"
    },
    {
      "id": 99,
      "name": "Documentary"
    },
    {
      "id": 18,
      "name": "Drama"
    },
    {
      "id": 10751,
      "name": "Family"
    },
    {
      "id": 14,
      "name": "Fantasy"
    },
    {
      "id": 36,
      "name": "History"
    },
    {
      "id": 27,
      "name": "Horror"
    },
    {
      "id": 10402,
      "name": "Music"
    },
    {
      "id": 9648,
      "name": "Mystery"
    },
    {
      "id": 10749,
      "name": "Romance"
    },
    {
      "id": 878,
      "name": "Science Fiction"
    },
    {
      "id": 10770,
      "name": "TV Movie"
    },
    {
      "id": 53,
      "name": "Thriller"
    },
    {
      "id": 10752,
      "name": "War"
    },
    {
  

## 2. Explore — Search a Movie
Search for a known movie by title and year. Inspect everything TMDB returns.

In [3]:
search_result = tmdb_get("search/movie", {"query": "Toy Story", "year": 1995})
print(json.dumps(search_result, indent=2))

{
  "page": 1,
  "results": [
    {
      "adult": false,
      "backdrop_path": "/3Rfvhy1Nl6sSGJwyjb0QiZzZYlB.jpg",
      "genre_ids": [
        10751,
        35,
        16,
        12
      ],
      "id": 862,
      "title": "Toy Story",
      "original_language": "en",
      "original_title": "Toy Story",
      "overview": "Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences.",
      "popularity": 17.9048,
      "poster_path": "/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg",
      "release_date": "1995-11-22",
      "softcore": false,
      "video": false,
      "vote_average": 7.972,
      "vote_count": 19793
    },
    {
      "adult": false,
      "backdrop_path": "/k7vO3lOOkxZ88Bieu5UQLEBKZOl.jpg",
      "genre_ids": [
        99
      ],
    

In [4]:
# Grab the TMDB ID from the top result — we'll use it in the next call
tmdb_id = search_result["results"][0]["id"]
print(f"TMDB ID for Toy Story: {tmdb_id}")

TMDB ID for Toy Story: 862


## 3. Explore — Movie Credits
Use the TMDB ID from above to fetch the full cast and crew. Inspect everything.

In [5]:
credits = tmdb_get(f"movie/{tmdb_id}/credits")
print(json.dumps(credits, indent=2))

{
  "id": 862,
  "cast": [
    {
      "adult": false,
      "gender": 2,
      "id": 31,
      "known_for_department": "Acting",
      "name": "Tom Hanks",
      "original_name": "Tom Hanks",
      "popularity": 12.6417,
      "profile_path": "/oFvZoKI6lvU03n4YoNGAll9rkas.jpg",
      "cast_id": 14,
      "character": "Woody (voice)",
      "credit_id": "52fe4284c3a36847f8024f95",
      "order": 0
    },
    {
      "adult": false,
      "gender": 2,
      "id": 12898,
      "known_for_department": "Acting",
      "name": "Tim Allen",
      "original_name": "Tim Allen",
      "popularity": 3.3918,
      "profile_path": "/woWhZzFILVhYMAvsPL171HjMY0y.jpg",
      "cast_id": 15,
      "character": "Buzz Lightyear (voice)",
      "credit_id": "52fe4284c3a36847f8024f99",
      "order": 1
    },
    {
      "adult": false,
      "gender": 2,
      "id": 7167,
      "known_for_department": "Acting",
      "name": "Don Rickles",
      "original_name": "Don Rickles",
      "popularity": 2.6658,


## 4. Load MovieLens `u.item`

In [6]:
import pandas as pd

# u.item is pipe-separated and uses Latin-1 encoding
columns = [
    "movieId", "title", "release_date", "video_release_date", "imdb_url",
    "unknown", "Action", "Adventure", "Animation", "Children", "Comedy",
    "Crime", "Documentary", "Drama", "Fantasy", "FilmNoir", "Horror",
    "Musical", "Mystery", "Romance", "SciFi", "Thriller", "War", "Western"
]

movies = pd.read_csv(
    "../data/raw/ml-100k/u.item",
    sep="|",
    names=columns,
    encoding="latin-1"
)

print(f"Shape: {movies.shape}")
movies[["movieId", "title", "imdb_url"]].head(10)

Shape: (1682, 24)


,movieId,title,imdb_url
0,1,Toy Story (1995),http://us.imdb.com/M/title-exact?Toy%20Story%2...
1,2,GoldenEye (1995),http://us.imdb.com/M/title-exact?GoldenEye%20(...
2,3,Four Rooms (1995),http://us.imdb.com/M/title-exact?Four%20Rooms%...
3,4,Get Shorty (1995),http://us.imdb.com/M/title-exact?Get%20Shorty%...
4,5,Copycat (1995),http://us.imdb.com/M/title-exact?Copycat%20(1995)
5,6,Shanghai Triad (Yao a yao yao dao waipo qiao) ...,http://us.imdb.com/Title?Yao+a+yao+yao+dao+wai...
6,7,Twelve Monkeys (1995),http://us.imdb.com/M/title-exact?Twelve%20Monk...
7,8,Babe (1995),http://us.imdb.com/M/title-exact?Babe%20(1995)
8,9,Dead Man Walking (1995),http://us.imdb.com/M/title-exact?Dead%20Man%20...
9,10,Richard III (1995),http://us.imdb.com/M/title-exact?Richard%20III...


In [7]:
# Inspect the IMDB URL format — this tells us if we can extract clean IDs
print("Sample IMDB URLs:")
for url in movies["imdb_url"].dropna().head(10):
    print(url)

Sample IMDB URLs:
http://us.imdb.com/M/title-exact?Toy%20Story%20(1995)
http://us.imdb.com/M/title-exact?GoldenEye%20(1995)
http://us.imdb.com/M/title-exact?Four%20Rooms%20(1995)
http://us.imdb.com/M/title-exact?Get%20Shorty%20(1995)
http://us.imdb.com/M/title-exact?Copycat%20(1995)
http://us.imdb.com/Title?Yao+a+yao+yao+dao+waipo+qiao+(1995)
http://us.imdb.com/M/title-exact?Twelve%20Monkeys%20(1995)
http://us.imdb.com/M/title-exact?Babe%20(1995)
http://us.imdb.com/M/title-exact?Dead%20Man%20Walking%20(1995)
http://us.imdb.com/M/title-exact?Richard%20III%20(1995)


## 5. Parse Title and Year

In [9]:
import re

def parse_title_year(raw_title):
    # Extract year from the last (YYYY) at end of string
    year_match = re.search(r'\((\d{4})\)\s*$', raw_title)
    parsed_year = int(year_match.group(1)) if year_match else None

    # Strip the (YYYY) from the title
    title = re.sub(r'\s*\(\d{4}\)\s*$', '', raw_title).strip()

    # Uninvert articles: "Lambs, The" → "The Lambs"
    for article in ["An", "The", "A"]:
        suffix = f", {article}"
        if title.endswith(suffix):
            title = f"{article} {title[:-len(suffix)]}"
            break

    return title, parsed_year

In [10]:
movies[["parsed_title", "parsed_year"]] = movies["title"].apply(
    lambda t: pd.Series(parse_title_year(t))
)

print(f"Total movies:    {len(movies)}")
print(f"Missing year:    {movies['parsed_year'].isna().sum()}")

# Show inverted article fixes
inverted = movies[movies["title"].str.contains(r', (The|An|A) \(', regex=True, na=False)]
print(f"Inverted titles: {len(inverted)}")
print()
display(inverted[["title", "parsed_title", "parsed_year"]].head(10))

print("\nGeneral sample:")
display(movies[["title", "parsed_title", "parsed_year"]].sample(10, random_state=42))

Total movies:    1682
Missing year:    2
Inverted titles: 364



C:\Users\Diego Sanchez\AppData\Local\Temp\ipykernel_31332\339595093.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  inverted = movies[movies["title"].str.contains(r', (The|An|A) \(', regex=True, na=False)]


,title,parsed_title,parsed_year
11,"Usual Suspects, The (1995)",The Usual Suspects,1995.0
17,"White Balloon, The (1995)",The White Balloon,1995.0
24,"Birdcage, The (1996)",The Birdcage,1996.0
25,"Brothers McMullen, The (1995)",The Brothers McMullen,1995.0
33,"Doom Generation, The (1995)",The Doom Generation,1995.0
37,"Net, The (1995)",The Net,1995.0
51,"Madness of King George, The (1994)",The Madness of King George,1994.0
54,"Professional, The (1994)",The Professional,1994.0
62,"Santa Clause, The (1994)",The Santa Clause,1994.0
63,"Shawshank Redemption, The (1994)",The Shawshank Redemption,1994.0



General sample:


,title,parsed_title,parsed_year
1393,Swept from the Sea (1997),Swept from the Sea,1997.0
743,Michael Collins (1996),Michael Collins,1996.0
1605,Deceiver (1997),Deceiver,1997.0
49,Star Wars (1977),Star Wars,1977.0
188,"Grand Day Out, A (1992)",A Grand Day Out,1992.0
482,Casablanca (1942),Casablanca,1942.0
892,For Richer or Poorer (1997),For Richer or Poorer,1997.0
613,Giant (1956),Giant,1956.0
1627,Lamerica (1994),Lamerica,1994.0
1346,"Ballad of Narayama, The (Narayama Bushiko) (1958)","Ballad of Narayama, The (Narayama Bushiko)",1958.0


## 6. Automated TMDB Search

In [14]:
import time

SAMPLE_SIZE = 5
SLEEP_SECONDS = 0.25

sample = movies.head(SAMPLE_SIZE).copy()
results = []

for _, row in sample.iterrows():
    year = int(row["parsed_year"]) if pd.notna(row["parsed_year"]) else None
    response = tmdb_get("search/movie", {"query": row["parsed_title"], "year": year})

    hits = response.get("results", [])
    if hits:
        top = hits[0]
        results.append({
            "movieId":    row["movieId"],
            "raw_title":  row["title"],
            "tmdb_id":    top["id"],
            "tmdb_title": top["title"],
            "tmdb_year":  top.get("release_date", "")[:4],
            "matched":    True,
        })
    else:
        results.append({
            "movieId":    row["movieId"],
            "raw_title":  row["title"],
            "tmdb_id":    None,
            "tmdb_title": None,
            "tmdb_year":  None,
            "matched":    False,
        })

    time.sleep(SLEEP_SECONDS)

sample_results = pd.DataFrame(results)
display(sample_results)
print(f"\nMatched: {sample_results['matched'].sum()} / {len(sample_results)}")

,movieId,raw_title,tmdb_id,tmdb_title,tmdb_year,matched
0,1,Toy Story (1995),862,Toy Story,1995,True
1,2,GoldenEye (1995),710,GoldenEye,1995,True
2,3,Four Rooms (1995),5,Four Rooms,1995,True
3,4,Get Shorty (1995),8012,Get Shorty,1995,True
4,5,Copycat (1995),1710,Copycat,1995,True



Matched: 5 / 5


In [22]:
SLEEP_SECONDS = 0.25
enriched = []
failed = []

for row in tqdm(movies.itertuples(), total=len(movies), desc="Enriching"):
    year = int(row.parsed_year) if pd.notna(row.parsed_year) else None

    # Step 1: search by title + year
    response = tmdb_get("search/movie", {"query": row.parsed_title, "year": year})
    time.sleep(SLEEP_SECONDS)

    hits = response.get("results", [])
    if not hits:
        print(f"  ✗ No match: {row.title}")
        record = {
            "movieId":   row.movieId,
            "raw_title": row.title,
            "tmdb_id":   None,
            "genre_ids": [],
            "cast":      [],
            "directors": [],
            "matched":   False,
        }
        enriched.append(record)
        failed.append({"movieId": row.movieId, "raw_title": row.title,
                        "parsed_title": row.parsed_title, "parsed_year": year})
        continue

    top = hits[0]

    # Step 2: fetch credits using the matched tmdb_id
    credits = tmdb_get(f"movie/{top['id']}/credits")
    time.sleep(SLEEP_SECONDS)

    enriched.append({
        "movieId":   row.movieId,
        "raw_title": row.title,
        "tmdb_id":   top["id"],
        "genre_ids": top.get("genre_ids", []),
        "cast":      [m["name"] for m in credits.get("cast", [])[:10]],
        "directors": [m["name"] for m in credits.get("crew", []) if m["job"] == "Director"],
        "matched":   True,
    })

enriched_df = pd.DataFrame(enriched)
failed_df = pd.DataFrame(failed)

print(f"\nMatched: {enriched_df['matched'].sum()} / {len(enriched_df)}")
print(f"Failed:  {len(failed_df)}")
display(enriched_df.head())

Enriching:   0%|          | 0/1682 [00:00<?, ?it/s]

  ✗ No match: Shanghai Triad (Yao a yao yao dao waipo qiao) (1995)
  ✗ No match: Dolores Claiborne (1994)
  ✗ No match: Brother Minister: The Assassination of Malcolm X (1994)
  ✗ No match: Return of the Pink Panther, The (1974)
  ✗ No match: Monty Python and the Holy Grail (1974)
  ✗ No match: Grand Day Out, A (1992)
  ✗ No match: Houseguest (1994)
  ✗ No match: Heavyweights (1994)
  ✗ No match: Red Rock West (1992)
  ✗ No match: Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963)
  ✗ No match: Stephen King's The Langoliers (1995)
  ✗ No match: Daniel Defoe's Robinson Crusoe (1996)
  ✗ No match: Conan the Barbarian (1981)
  ✗ No match: Canadian Bacon (1994)
  ✗ No match: Mute Witness (1994)
  ✗ No match: Beyond Bedlam (1993)
  ✗ No match: Swimming with Sharks (1995)
  ✗ No match: Boys Life (1995)
  ✗ No match: Power 98 (1995)
  ✗ No match: Loch Ness (1995)
  ✗ No match: Two or Three Things I Know About Her (1966)
  ✗ No match: Nosferatu a Venezia (1986)
  ✗ No 

,movieId,raw_title,tmdb_id,genre_ids,cast,directors,matched
0,1,Toy Story (1995),862.0,"[10751, 35, 16, 12]","[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...",[John Lasseter],True
1,2,GoldenEye (1995),710.0,"[12, 28, 53]","[Pierce Brosnan, Sean Bean, Izabella Scorupco,...",[Martin Campbell],True
2,3,Four Rooms (1995),5.0,[35],"[Tim Roth, Jennifer Beals, David Proval, Ione ...","[Allison Anders, Alexandre Rockwell, Quentin T...",True
3,4,Get Shorty (1995),8012.0,"[35, 53, 80]","[John Travolta, Gene Hackman, Rene Russo, Dann...",[Barry Sonnenfeld],True
4,5,Copycat (1995),1710.0,"[53, 80, 9648]","[Sigourney Weaver, Holly Hunter, Dermot Mulron...",[Jon Amiel],True


In [25]:
SLEEP_SECONDS = 0.25
retry_enriched = []
still_failed = []

for row in tqdm(failed_df.itertuples(), total=len(failed_df), desc="Retrying"):
    # Search by title only — drop the year constraint
    response = tmdb_get("search/movie", {"query": row.parsed_title})
    time.sleep(SLEEP_SECONDS)

    hits = response.get("results", [])
    if not hits:
        print(f"  ✗ Still no match: {row.raw_title}")
        still_failed.append({"movieId": row.movieId, "raw_title": row.raw_title,
                              "parsed_title": row.parsed_title, "parsed_year": row.parsed_year})
        continue

    top = hits[0]
    credits = tmdb_get(f"movie/{top['id']}/credits")
    time.sleep(SLEEP_SECONDS)

    print(f"  ✓ Recovered: {row.raw_title}  →  {top['title']} ({top.get('release_date', '')[:4]})")
    retry_enriched.append({
        "movieId":   row.movieId,
        "raw_title": row.raw_title,
        "tmdb_id":   top["id"],
        "genre_ids": top.get("genre_ids", []),
        "cast":      [m["name"] for m in credits.get("cast", [])[:10]],
        "directors": [m["name"] for m in credits.get("crew", []) if m["job"] == "Director"],
        "matched":   True,
    })

# Merge retried successes back into enriched_df
if retry_enriched:
    retried_df = pd.DataFrame(retry_enriched)
    recovered_ids = set(retried_df["movieId"])
    enriched_df = pd.concat([
        enriched_df[~enriched_df["movieId"].isin(recovered_ids)],
        retried_df
    ]).sort_values("movieId").reset_index(drop=True)

failed_df = pd.DataFrame(still_failed) if still_failed else pd.DataFrame()

print(f"\nRecovered: {len(retry_enriched)}")
print(f"Still failed: {len(failed_df)}")
print(f"Total matched: {enriched_df['matched'].sum()} / {len(enriched_df)}")

Retrying:   1%|▏         | 1/78 [00:00<00:54,  1.41it/s]

  ✗ Still no match: Shanghai Triad (Yao a yao yao dao waipo qiao) (1995)


Retrying:   3%|▎         | 2/78 [00:02<01:23,  1.10s/it]

  ✓ Recovered: Dolores Claiborne (1994)  →  Dolores Claiborne (1995)


Retrying:   4%|▍         | 3/78 [00:03<01:29,  1.19s/it]

...

  ✗ Still no match: Bitter Sugar (Azucar Amargo) (1996)


Retrying:  99%|█████████▊| 77/78 [01:05<00:00,  1.12it/s]

  ✓ Recovered: Next Step, The (1995)  →  The Next Step (1997)


Retrying: 100%|██████████| 78/78 [01:06<00:00,  1.17it/s]

  ✓ Recovered: Further Gesture, A (1996)  →  A Further Gesture (1997)

Recovered: 62
Still failed: 16
Total matched: 1666 / 1682


## 7. Failed Matches

In [29]:
import os

os.makedirs("../data/enriched", exist_ok=True)
failed_df.to_csv("../data/enriched/failed_matches.csv", index=False)

print(f"Exported {len(failed_df)} failed matches → data/enriched/failed_matches.csv")
display(failed_df)

Exported 16 failed matches → data/enriched/failed_matches.csv


,movieId,raw_title,parsed_title,parsed_year
0,6,Shanghai Triad (Yao a yao yao dao waipo qiao) ...,Shanghai Triad (Yao a yao yao dao waipo qiao),1995.0
1,563,Stephen King's The Langoliers (1995),Stephen King's The Langoliers,1995.0
2,912,U.S. Marshalls (1998),U.S. Marshalls,1998.0
3,1201,Marlene Dietrich: Shadow and Light (1996),Marlene Dietrich: Shadow and Light,1996.0
4,1235,"Big Bang Theory, The (1994)",The Big Bang Theory,1994.0
5,1345,"Day the Sun Turned Cold, The (Tianguo niezi) (...","Day the Sun Turned Cold, The (Tianguo niezi)",1994.0
6,1359,Boys in Venice (1996),Boys in Venice,1996.0
7,1405,Boy's Life 2 (1997),Boy's Life 2,1997.0
8,1412,Land Before Time III: The Time of the Great Gi...,Land Before Time III: The Time of the Great Gi...,NaN
9,1420,Gilligan's Island: The Movie (1998),Gilligan's Island: The Movie,1998.0


## 8. Manual Mapping

In [32]:
# Movies still needing a manual TMDB ID
display(failed_df[["movieId", "raw_title", "parsed_title"]].reset_index(drop=True))

,movieId,raw_title,parsed_title
0,6,Shanghai Triad (Yao a yao yao dao waipo qiao) ...,Shanghai Triad (Yao a yao yao dao waipo qiao)
1,563,Stephen King's The Langoliers (1995),Stephen King's The Langoliers
2,912,U.S. Marshalls (1998),U.S. Marshalls
3,1201,Marlene Dietrich: Shadow and Light (1996),Marlene Dietrich: Shadow and Light
4,1235,"Big Bang Theory, The (1994)",The Big Bang Theory
5,1345,"Day the Sun Turned Cold, The (Tianguo niezi) (...","Day the Sun Turned Cold, The (Tianguo niezi)"
6,1359,Boys in Venice (1996),Boys in Venice
7,1405,Boy's Life 2 (1997),Boy's Life 2
8,1412,Land Before Time III: The Time of the Great Gi...,Land Before Time III: The Time of the Great Gi...
9,1420,Gilligan's Island: The Movie (1998),Gilligan's Island: The Movie


In [65]:
# Look up each movie at themoviedb.org — the ID is the number in the URL:
# themoviedb.org/movie/862-toy-story  →  862
#
# Fill in one entry per movie, then run this cell.

manual_mapping = [
    {"movieId": 6, "tmdb_id": 37557},
    {"movieId": 912, "tmdb_id": 11808},
    {"movieId": 1201, "tmdb_id": 581579},
    {"movieId": 1345, "tmdb_id": 48144},
    {"movieId": 1405, "tmdb_id": 50091},
    {"movieId": 1412, "tmdb_id": 19004},
    {"movieId": 1420, "tmdb_id": 32677},
    {"movieId": 1536, "tmdb_id": 11985},
    {"movieId": 1569, "tmdb_id": 202425},
    {"movieId": 1591, "tmdb_id": 11220},
    {"movieId": 1630, "tmdb_id": 44284},
    {"movieId": 1633, "tmdb_id": 68445},
    {"movieId": 1634, "tmdb_id": 117730},
    {"movieId": 1639, "tmdb_id": 43777},    
]

In [66]:
manual_enriched = []

for entry in tqdm(manual_mapping, desc="Manual mapping"):
    tmdb_id = entry["tmdb_id"]
    movie   = tmdb_get(f"movie/{tmdb_id}")
    time.sleep(SLEEP_SECONDS)
    credits = tmdb_get(f"movie/{tmdb_id}/credits")
    time.sleep(SLEEP_SECONDS)

    print(f"  ✓ {movie['title']} ({movie.get('release_date', '')[:4]})")
    manual_enriched.append({
        "movieId":   entry["movieId"],
        "raw_title": movie["title"],
        "tmdb_id":   tmdb_id,
        "genre_ids": [g["id"] for g in movie.get("genres", [])],
        "cast":      [m["name"] for m in credits.get("cast", [])[:10]],
        "directors": [m["name"] for m in credits.get("crew", []) if m["job"] == "Director"],
        "matched":   True,
    })

# Merge back into enriched_df
manual_df = pd.DataFrame(manual_enriched)
mapped_ids = set(manual_df["movieId"])
enriched_df = pd.concat([
    enriched_df[~enriched_df["movieId"].isin(mapped_ids)],
    manual_df
]).sort_values("movieId").reset_index(drop=True)

print(f"\nManually mapped: {len(manual_enriched)}")
print(f"Total matched:   {enriched_df['matched'].sum()} / {len(enriched_df)}")

Manual mapping:   7%|▋         | 1/14 [00:00<00:10,  1.19it/s]

  ✓ Shanghai Triad (1995)


Manual mapping:  14%|█▍        | 2/14 [00:01<00:09,  1.21it/s]

  ✓ U.S. Marshals (1998)


Manual mapping:  21%|██▏       | 3/14 [00:02<00:09,  1.22it/s]

...

  ✓ Cold Fever (1995)


Manual mapping:  93%|█████████▎| 13/14 [00:10<00:00,  1.23it/s]

  ✓ Under The Domim Tree (1994)


Manual mapping: 100%|██████████| 14/14 [00:11<00:00,  1.23it/s]

  ✓ Bitter Sugar (1996)

Manually mapped: 14
Total matched:   1679 / 1682


## 9. Save Enriched Dataset

In [72]:
# Build genre id → name lookup from Section 1
genre_map = {g["id"]: g["name"] for g in genres["genres"]}

enriched_df["genre_names"] = enriched_df["genre_ids"].apply(
    lambda ids: [genre_map.get(i, str(i)) for i in ids]
)

# Save
os.makedirs("../data/enriched", exist_ok=True)
enriched_df.to_csv("../data/enriched/enriched_movies.csv", index=False)

total   = len(enriched_df)
matched = int(enriched_df["matched"].sum())
print(f"Saved → data/enriched/enriched_movies.csv")
print(f"Match rate: {matched} / {total} ({matched / total:.1%})")
display(enriched_df[["movieId", "raw_title", "tmdb_id", "genre_ids", "genre_names", "cast", "directors"]].head(5))

Saved → data/enriched/enriched_movies.csv
Match rate: 1679 / 1682 (99.8%)


,movieId,raw_title,tmdb_id,genre_ids,genre_names,cast,directors
0,1,Toy Story (1995),862.0,"[10751, 35, 16, 12]","[Family, Comedy, Animation, Adventure]","[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...",[John Lasseter]
1,2,GoldenEye (1995),710.0,"[12, 28, 53]","[Adventure, Action, Thriller]","[Pierce Brosnan, Sean Bean, Izabella Scorupco,...",[Martin Campbell]
2,3,Four Rooms (1995),5.0,[35],[Comedy],"[Tim Roth, Jennifer Beals, David Proval, Ione ...","[Allison Anders, Alexandre Rockwell, Quentin T..."
3,4,Get Shorty (1995),8012.0,"[35, 53, 80]","[Comedy, Thriller, Crime]","[John Travolta, Gene Hackman, Rene Russo, Dann...",[Barry Sonnenfeld]
4,5,Copycat (1995),1710.0,"[53, 80, 9648]","[Thriller, Crime, Mystery]","[Sigourney Weaver, Holly Hunter, Dermot Mulron...",[Jon Amiel]
